# ProCIR training on a Jupyter GPU server

**Run All, any time.** The notebook drives the whole pipeline and every cell is safe to re-run:
the repo is pulled, packages are reinstalled after a pod restart, extracted data is only verified, the
speed pilot's result is reused, a running training queue is left alone, a stopped run resumes from its
last checkpoint and finished runs are skipped. Training itself runs in the background (`nohup`) through
the repo's scripts, so it survives browser disconnects and kernel restarts.

1. **Parameters**: paste the Google Drive folder link once, for the first download.
2. **Setup**: repo, packages, data, GPU check, GPU tests.
3. **Speed pilot** (~40 min, once): a few training steps per `SPEED_CANDIDATES` setting; the fastest one
   that reserves at most `VRAM_BUDGET_GB` is kept in `runs/speed.json`. Batch size stays 64, so the
   choice changes speed and memory, not results.
4. **Launch**: the runs in `RUNS` go into one background queue: train → export → eval with the
   unmodified upstream `evaluate.py`.
5. **Monitor**, **Results**, **Stop**: re-run any time (Stop only after setting `CONFIRM = True`).

In [ ]:
# ---- parameters ----
import os
REPO_URL = "https://github.com/vbphuong18/HCMUT_Fashion_MV.git"
REPO_DIR = os.path.expanduser("~/HCMUT_Fashion_MV")  # home needs ~45 GB: create the notebook with a 100 Gi workspace volume
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/PASTE_FOLDER_ID_HERE"  # only for the first download
CONFIG = "configs/procir_fast.yaml"     # MT+Align, all three sources, 3 views at 336^2, LoRA, B=64
VRAM_BUDGET_GB = 8.0                    # GPU memory training may reserve (the lab's MIG slice has ~11 GiB free)
SPEED_CANDIDATES = [(4, True), (8, True), (4, False), (8, False)]  # (chunk_size, gradient checkpointing)
RERUN_PILOT = False                     # True: measure again (stops a running queue; Launch resumes it)
TARGET_TRAIN_HOURS = 12.0               # training time per run, used to pick train_fraction for a new run
MAX_TRAIN_FRACTION = 0.25               # the proposal's upper bound
EVAL_BATCH_SIZE = 16                    # dev eval during training and the final eval; the GPU check lowers it
RUNS = [                                # (output name, extra config overrides); run in this order
    ("fast_mt_align_s42", []),
    ("fast_single_s42", ["multi_turn=false", "align=false"]),   # single-turn baseline (paper Table 3, row 1)
]
EVAL_PROTOCOLS = ["upstream", "train"]  # upstream = 5 views / 512^2 (reported); train = 3 views / 336^2
EXPORT_CHECKPOINT = "best"             # best = highest dev mean R@5 (ckpt/best.pt); "latest" = final step
RUN_GPU_TESTS = True                    # skipped while a training queue is running

In [ ]:
import json
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command and stream its output into the notebook."""
    print("+", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (REPO_DIR if Path(REPO_DIR).exists() else None),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")
    return p.returncode


PY = shlex.quote(sys.executable)
ENV = {"PYTHONPATH": "src:external/FashionMV", "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}  # less fragmentation when GPU memory is tight


def queue_running():
    # "[q]" keeps the pattern from matching the shell that runs pgrep
    return subprocess.run(["pgrep", "-f", "runs/[q]ueue.sh"], capture_output=True).returncode == 0


def stop_queue():
    """Stop the background queue and its training; the next Launch resumes from the last checkpoint."""
    sh("pkill -f 'runs/[q]ueue.sh'; pkill -f 'procir_[t]rain'; sleep 10; "
       "pgrep -af 'procir_[t]rain' || echo 'training queue stopped'", check=False)

## 1. Setup

In [ ]:
# ---- code: clone or update the repo ----
NB = "src/server/procir_server.ipynb"
if Path(REPO_DIR, ".git").exists():
    os.chdir(REPO_DIR)
    # this notebook is saved with its outputs and the Drive link: set that copy aside so the pull is not blocked
    if sh(f"git diff --quiet -- {NB}", check=False):
        sh(f"git stash push -q -m 'notebook before pull' -- {NB}")
    before = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    sh("git pull --ff-only && git submodule update --init")
    if sh(f"git diff --quiet {before} HEAD -- {NB}", check=False):
        raise RuntimeError("this notebook changed upstream: File > Reload Notebook from Disk, then Run All again")
else:
    sh(f"git clone --recurse-submodules {REPO_URL} {shlex.quote(REPO_DIR)}", cwd=os.path.expanduser("~"))
    os.chdir(REPO_DIR)
sh("git log --oneline -1 && git -C external/FashionMV rev-parse --short HEAD")

In [ ]:
# ---- packages; the linear-attention kernels one at a time so one failed build cannot block the rest ----
sh(f"{PY} -m pip install -q -r requirements-train.txt gdown")
sh(f"{PY} -m pip install -q flash-linear-attention", check=False)
# kubeflow cuda images: conda's scipy/sklearn need a newer libstdc++ (CXXABI_1.3.15) than the system one the
# process loads, so `import transformers` fails; the pip wheels do not need it
if sh(f'{PY} -c "import torch, scipy.spatial, sklearn"', check=False) != 0:
    sh(f"{PY} -m pip install -q --force-reinstall --no-deps scipy scikit-learn")
    sh(f'{PY} -c "import torch, scipy.spatial, sklearn, transformers, peft"')
# flash-linear-attention runs Triton kernels, and Triton compiles a small C launcher at run time: the image has
# no gcc and no sudo, so take it from conda-forge
if not (shutil.which("gcc") or shutil.which("cc")):
    sh(f"{shlex.quote(str(Path(sys.executable).parent / 'conda'))} install -y -q -c conda-forge gcc gxx")
# fla refuses Triton 3.4-3.7.0 on Hopper (H200): wrong gradients in gated chunk_bwd_dqkwg (fla #640).
# 3.7.1 is the patch release torch 2.12.1 pins; pip warns that torch 2.12.0 asks for 3.7.0
from importlib.metadata import version
from packaging.version import Version
if Version("3.4.0") <= Version(version("triton")) < Version("3.7.1"):
    sh(f"{PY} -m pip install -q triton==3.7.1")
has_nvcc = sh("nvcc --version", check=False) == 0
if has_nvcc:
    sh(f"MAX_JOBS=8 {PY} -m pip install -q causal-conv1d --no-build-isolation", check=False)
else:
    print("no nvcc: causal-conv1d stays on its PyTorch fallback (slower convolution only)")

In [ ]:
# ---- data: download the zips from Drive once, extract into data/, check every triplet has images ----
fetched = Path("data/.fetched")
if "PASTE_FOLDER_ID" in GDRIVE_FOLDER and fetched.is_dir() and any(fetched.iterdir()):
    print("data already extracted:", sorted(m.name for m in fetched.iterdir()))
    sh(f"{PY} scripts/setup_data.py verify")
else:
    # one zip at a time (download, extract, delete): needs ~19 GB for the data plus ~4 GB for one zip
    sh("df -h . ~", check=False)
    sh(f"{PY} scripts/fetch_data.py --gdrive-folder {shlex.quote(GDRIVE_FOLDER)}")

In [ ]:
# ---- GPU check: strict first, then accept missing kernels (slower) ----
ALLOW_SLOW = sh(f"{PY} scripts/check_env.py runs/env_check.json", check=False) != 0
if ALLOW_SLOW:
    sh(f"{PY} scripts/check_env.py --allow-slow runs/env_check.json")
import torch
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free {free / 2**30:.1f} / {total / 2**30:.1f} GiB (a running training queue counts as used)")
if free < 18 * 2**30 or queue_running():  # the lab server: a shared MIG slice with ~11 GiB left
    print("less than 18 GiB free: EVAL_BATCH_SIZE = 8 (the final eval runs 5 views at 512^2)")
    EVAL_BATCH_SIZE = min(EVAL_BATCH_SIZE, 8)
if VRAM_BUDGET_GB > free / 2**30 and not queue_running():
    print(f"WARNING: VRAM_BUDGET_GB={VRAM_BUDGET_GB} is more than the {free / 2**30:.1f} GiB free")

In [ ]:
# ---- GPU tests (a few minutes): export matches upstream evaluate.py, padding invariance, resume ----
if RUN_GPU_TESTS and not queue_running():
    sh(f"{PY} -m pytest -q -m gpu -x", env=ENV)
else:
    print("GPU tests skipped" + (" (a training queue is running)" if queue_running() else ""))

## 2. Speed pilot: the fastest setting within `VRAM_BUDGET_GB` → train_fraction

In [ ]:
import yaml


def pipeline_cmd(name, extra, chunk, gc, stages=("train", "export", "eval")):
    sets = [f"output_dir=runs/{name}", f"chunk_size={chunk}", f"gradient_checkpointing={str(gc).lower()}",
            f"eval_batch_size={EVAL_BATCH_SIZE}", *extra]
    flags = " --allow-slow" if ALLOW_SLOW else ""
    return (f"{PY} -m procir_train.pipeline --config {CONFIG} --set {' '.join(sets)} "
            f"--stages {' '.join(stages)} --eval-protocols {' '.join(EVAL_PROTOCOLS)} "
            f"--eval-batch-size {EVAL_BATCH_SIZE} --export-checkpoint {EXPORT_CHECKPOINT}{flags}")


def train_pool(run):
    stats = json.loads(Path("runs", run, "data_stats.json").read_text())
    return sum(v["train_pool"] for v in stats.values() if isinstance(v, dict) and "train_pool" in v)


SPEED = Path("runs/speed.json")
if SPEED.exists() and not RERUN_PILOT:
    best = json.loads(SPEED.read_text())
    print("using the measured setting in runs/speed.json:", {k: v for k, v in best.items() if k != "trials"})
else:
    if queue_running():
        print("stopping the training queue to measure; Launch resumes it from its last checkpoint")
        stop_queue()
    trials = []
    for chunk, gc in SPEED_CANDIDATES:
        name = f"pilot_c{chunk}_gc{int(gc)}"
        sh(f"rm -rf runs/{name}")
        # 12 steps (the first 5 are warm-up and not timed) and one small dev eval to check its memory too
        ok = sh(pipeline_cmd(name, ["stop_after_steps=12", "log_every=1", "eval_every=12",
                                    "dev_eval_max_queries=64", "save_every=100000"], chunk, gc, stages=("train",)),
                check=False, env=ENV) == 0
        s = json.loads(subprocess.run([sys.executable, "scripts/summarize_run.py", f"runs/{name}"],
                                      capture_output=True, text=True).stdout or "{}")
        mem, sec = s.get("max_mem_reserved_gb"), s.get("sec_per_step_median")
        fits = ok and sec is not None and mem is not None and mem <= VRAM_BUDGET_GB
        trials.append({"chunk_size": chunk, "gradient_checkpointing": gc, "ok": ok, "sec_per_step": sec,
                       "max_mem_reserved_gb": mem, "fits": fits, "run": name})
        print(f"==> chunk_size={chunk} checkpointing={gc}: ok={ok} sec/step={sec} reserved={mem} GB fits={fits}")
    good = [t for t in trials if t["fits"]]
    if not good:
        raise RuntimeError(f"no setting finished within {VRAM_BUDGET_GB} GB; see the log above: {trials}")
    best = dict(min(good, key=lambda t: t["sec_per_step"]), trials=trials)
    SPEED.write_text(json.dumps(best, indent=2))
    print("fastest within budget:", {k: v for k, v in best.items() if k != "trials"})

sec, pool = best["sec_per_step"], train_pool(best["run"])
started = Path("runs", RUNS[0][0], "config.yaml")
if started.exists():  # a started run must resume on the same data
    TRAIN_FRACTION = yaml.safe_load(started.read_text())["train_fraction"]
    print(f"train_fraction={TRAIN_FRACTION}, kept from the started run {RUNS[0][0]}")
else:
    TRAIN_FRACTION = round(min(MAX_TRAIN_FRACTION, int(TARGET_TRAIN_HOURS * 3600 / sec) * 64 / pool), 3)
    print(f"train pool {pool} triplets -> train_fraction={TRAIN_FRACTION}; "
          f"MAX_TRAIN_FRACTION would take ~{MAX_TRAIN_FRACTION * pool / 64 * sec / 3600:.1f} h per run")
print(f"chunk_size={best['chunk_size']} checkpointing={best['gradient_checkpointing']} sec/step={sec:.1f}: "
      f"~{int(TRAIN_FRACTION * pool / 64)} steps, ~{TRAIN_FRACTION * pool / 64 * sec / 3600:.1f} h of training per run")

## 3. Launch all runs in a background queue

In [ ]:
def run_done(name, extra):
    """All eval outputs of a run exist (multi-turn runs are scored by upstream evaluate.py)."""
    mt = "multi_turn=false" not in extra
    need = [("eval_upstream/eval_results.json" if mt else "eval_upstream.json") if p == "upstream"
            else f"eval_{p}.json" for p in EVAL_PROTOCOLS]
    return all(Path("runs", name, n).exists() for n in need)


if queue_running():
    print("a training queue is already running: nothing launched (see Monitor)")
else:
    Path("runs").mkdir(exist_ok=True)
    lines = ["#!/usr/bin/env bash", f"cd {shlex.quote(REPO_DIR)}",
             "export " + " ".join(f"{k}={v}" for k, v in ENV.items())]
    for name, extra in RUNS:
        if run_done(name, extra):
            print(f"{name}: finished, skipped")
            continue
        # the pipeline resumes from runs/<name>/ckpt/latest.pt when it exists
        cmd = pipeline_cmd(name, [f"train_fraction={TRAIN_FRACTION}", *extra],
                           best["chunk_size"], best["gradient_checkpointing"])
        lines += [f'echo "=== {name} start $(date)"', f"{cmd} >> runs/{name}.log 2>&1",
                  f'echo "=== {name} exit $? $(date)"']
    Path("runs/queue.sh").write_text("\n".join(lines) + "\n")
    print(Path("runs/queue.sh").read_text())
    sh("nohup bash runs/queue.sh >> runs/queue.log 2>&1 &")
    time.sleep(5)
    print("queue running:", queue_running())

## 4. Monitor (re-run any time)

In [ ]:
print("queue running:", queue_running())
sh("cat runs/queue.log; nvidia-smi | grep -E 'MiB /'", check=False)  # MIG: the whole slice
for name, _ in RUNS:
    run = Path("runs", name)
    print(f"\n## {name}:", (run / "STATUS").read_text().strip() if (run / "STATUS").exists() else "")
    log = run / "train_log.jsonl"
    if log.exists():
        rows = [json.loads(l) for l in log.read_text().splitlines() if l.strip()]
        for r in [r for r in rows if "loss" in r][-3:]:
            print({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()
                   if k in ("step", "loss", "l_cir", "l_doc", "l_src", "lr", "grad_norm", "sec_per_step", "max_mem_reserved_gb")})
        for r in [r for r in rows if "dev" in r][-1:]:
            print("dev:", r)
    if Path("runs", f"{name}.log").exists():
        sh(f"tail -n 3 runs/{name}.log", check=False)

## 5. Results

In [ ]:
table = []
for name, _ in RUNS:
    up = Path("runs", name, "eval_upstream", "eval_results.json")
    ours = Path("runs", name, "eval_upstream.json")  # single-turn runs are scored by retrieval_eval
    res = json.loads(up.read_text()) if up.exists() else (json.loads(ours.read_text()) if ours.exists() else None)
    tr = Path("runs", name, "eval_train.json")
    table.append((name, res, json.loads(tr.read_text()) if tr.exists() else None))
for name, res, tr in table:
    print(f"\n### {name}")
    print("upstream protocol (5 views, 512^2):", json.dumps(res, indent=1) if res else "not yet")
    print("train protocol (3 views, 336^2):", json.dumps(tr, indent=1) if tr else "not yet")

In [ ]:
# ---- bundle small result files to download (no checkpoints) ----
sh("tar -czf runs/results_bundle.tgz runs/*.log runs/env_check.json runs/speed.json "
   "runs/*/config.yaml runs/*/train_log.jsonl runs/*/data_stats.json runs/*/env.json runs/*/STATUS "
   "runs/*/eval_upstream runs/*/eval_*.json 2>/dev/null; ls -la runs/results_bundle.tgz", check=False)

## 6. Stop the training queue

In [ ]:
# ---- stop training: set CONFIRM = True and run; Launch (or Run All) resumes from the last checkpoint ----
CONFIRM = False
if CONFIRM:
    stop_queue()
else:
    print("not stopped: set CONFIRM = True to stop; queue running:", queue_running())